In [25]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy.linalg import solve_triangular

In [26]:
fh = "prostate.csv"
df = pd.read_csv(fh)

df.sample(n=3)

,lcavol,lweight,age,lbph,svi,lcp,gleason,pgg45,lpsa
14,1.205971,3.4420,57,-1.386294,0,-0.43078,7,5,1.39872
81,2.513656,3.4735,57,0.438255,0,2.32728,7,60,3.53076
36,1.423108,3.6571,73,-0.579818,0,1.65823,8,15,2.15756


### Fitting w/ NumPy

In [27]:
# Define response and predictors

x_cols = df.columns.drop('lpsa').to_list()

y = df.loc[:, 'lpsa'].values
X = df.loc[:, x_cols].values

X[:3, :]

array([[-0.5798185,  2.7695   , 50.       , -1.386294 ,  0.       ,
        -1.38629  ,  6.       ,  0.       ],
       [-0.9942523,  3.3196   , 58.       , -1.386294 ,  0.       ,
        -1.38629  ,  6.       ,  0.       ],
       [-0.5108256,  2.6912   , 74.       , -1.386294 ,  0.       ,
        -1.38629  ,  7.       , 20.       ]])

In [28]:
X.shape

(97, 8)

In [29]:
# Add an intercept

X = np.hstack(
    [np.ones(shape=(X.shape[0], 1)), X]
)

X[:3, :]

array([[ 1.       , -0.5798185,  2.7695   , 50.       , -1.386294 ,
         0.       , -1.38629  ,  6.       ,  0.       ],
       [ 1.       , -0.9942523,  3.3196   , 58.       , -1.386294 ,
         0.       , -1.38629  ,  6.       ,  0.       ],
       [ 1.       , -0.5108256,  2.6912   , 74.       , -1.386294 ,
         0.       , -1.38629  ,  7.       , 20.       ]])

In [30]:
# Use NumPy's OLS function

beta, _, _, _ = np.linalg.lstsq(X, y, rcond=None)

beta

array([ 0.6693367 ,  0.58702183,  0.45446742, -0.01963718,  0.10705403,
        0.76615733, -0.10547426,  0.0451416 ,  0.00452523])

In [31]:
# Coeffiecients

x_cols_w_int = ['intercept'] + x_cols

res = pd.DataFrame(
    data = {c: [b] for c, b in zip(x_cols_w_int, beta)},
    index = ['coef']
)

res

,intercept,lcavol,lweight,age,lbph,svi,lcp,gleason,pgg45
coef,0.669337,0.587022,0.454467,-0.019637,0.107054,0.766157,-0.105474,0.045142,0.004525


### Fitting w/ Algorithm 3.1

In [32]:
# Define algorithm 3.1

def algorithm_3_1_all_coef(X: np.ndarray, y: np.ndarray) -> np.ndarray:
    """Returns all coefficients in a computationally efficient way."""
    n, p = X.shape
    Z = np.empty(shape=(n, p + 1))
    Z[:, 0] = 1.0
    D = np.zeros(shape=(p + 1, p + 1))      # X = Z D^{-1} D G
    G = np.eye(p + 1)
    D[0,0] = np.sqrt(n)

    for j in range(1, p + 1):
        z_j = X[:, j - 1].astype(float)

        for l in range(j):
            z_l   = Z[:, l]
            gamma = np.dot(z_j, z_l) / np.dot(z_l, z_l)
            G[l, j] = gamma
            z_j -= gamma * z_l

        z_norm = np.linalg.norm(z_j)
        Z[:, j] = z_j
        D[j,j] = z_norm

    Q = Z / np.diag(D)
    R = D @ G

    return solve_triangular(R, Q.T@y, lower=False)

In [33]:
# Define response and predictors

y = df.loc[:, 'lpsa'].values
X = df.loc[:, x_cols].values

X[:3, :]

array([[-0.5798185,  2.7695   , 50.       , -1.386294 ,  0.       ,
        -1.38629  ,  6.       ,  0.       ],
       [-0.9942523,  3.3196   , 58.       , -1.386294 ,  0.       ,
        -1.38629  ,  6.       ,  0.       ],
       [-0.5108256,  2.6912   , 74.       , -1.386294 ,  0.       ,
        -1.38629  ,  7.       , 20.       ]])

In [34]:
# All Coeffiecients

coefs = algorithm_3_1_all_coef(X, y)

a = pd.DataFrame(
    data = {c: [b] for c, b in zip(x_cols_w_int, coefs)},
    index = ['coef']
)

a

,intercept,lcavol,lweight,age,lbph,svi,lcp,gleason,pgg45
coef,0.669337,0.587022,0.454467,-0.019637,0.107054,0.766157,-0.105474,0.045142,0.004525


### Fitting w/ Statsmodels (extra)

In [35]:
X = df.drop(columns='lpsa')
y = df.loc[:, 'lpsa']

X = sm.add_constant(X)
model = sm.OLS(y, X).fit()

params = model.params

params.name = 'coef'
params = params.rename(index={'const': 'intercept'})

res = pd.DataFrame(
    data = {c: [b] for c, b in params.items()},
    index = ['coef']
)

res

,intercept,lcavol,lweight,age,lbph,svi,lcp,gleason,pgg45
coef,0.669337,0.587022,0.454467,-0.019637,0.107054,0.766157,-0.105474,0.045142,0.004525
